# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv

import chromadb
from chromadb.utils import embedding_functions

In [2]:
load_dotenv()

GAMES_DIR = Path("./games/")
CHROMA_PATH = "chroma_db"
COLLECTION_NAME = "games_collection"

### VectorDB Instance

In [3]:
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

### Collection

In [4]:
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME, embedding_function=embedding_fn)

c:\Users\laris\OneDrive\Documents\Studies\data_science\projetos\1777071084\project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9363.50it/s]


### Add documents

In [5]:
for file_name in sorted(os.listdir(GAMES_DIR)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(GAMES_DIR, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

In [6]:
results = collection.query(
    query_texts=["Which game was released for Nintendo 64?"],
    n_results=3
)

for doc in results["documents"][0]:
    print(doc)
    print("-" * 50)

[Nintendo 64] Super Mario 64 (1996) - A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.
--------------------------------------------------
[Super Nintendo Entertainment System (SNES)] Super Mario World (1990) - A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur Land from Bowser.
--------------------------------------------------
[GameCube] Super Smash Bros. Melee (2001) - A crossover fighting game featuring characters from various Nintendo franchises battling it out in dynamic arenas.
--------------------------------------------------
